In [1]:
!pip install -U transformers datasets peft accelerate bitsandbytes trl

Defaulting to user installation because normal site-packages is not writeable
  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl.metadata (10 kB)
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
    --------------------------------------- 0.3/12.8 MB ? eta -:--:--
    --------------------------------------- 0.3/12.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.8 MB 1.8 MB/s eta 0:00:07
   ---- ----------------------------------- 1.3/12.8 MB 1.8 MB/s eta 0:00:07
   ----- ---------------------------------- 1.8/12.8 MB 1.7 MB/s eta 0:00:07
   ------ --------------------------------- 2.1/12.8 MB 1.5 MB/s eta 0:00:08
   ------- -------------------------------- 2.4/12.8 MB 1.6 MB/s eta 0:00:07
   -------- ------------------------------- 2.6/12.8 MB 1.5 MB/s eta 0:00:07
   --------- ------------------------------ 3.1/12.8 MB 1.7 MB/s eta 0:00:06
   ---------- --------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.3.0 which is incompatible.
sentence-transformers 6.1.0 requires huggingface-hub<2.0.0,>=1.3.0, but you have huggingface-hub 2.3.0 which is incompatible.
unsloth 2026.10.6 requires datasets!=4.0.*,!=4.1.0,!=4.4.*,!=4.5.0,<5.0.0,>=3.4.1, but you have datasets 5.1.0 which is incompatible.
unsloth 2026.10.6 requires transformers!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.17.0,>=4.52.4, but you have transformers 5.20.2 which is incompatible.
unsloth 2026.10.6 requires trl!=0.19.0,<=1.15.0,>=0.18.2, but you have trl 1.16.0 which is incompatible.
unsloth-zoo 2026.10.6 requires datasets!=4.0.*,!=4.1.0,!=4.4.*,!=4.5.0,<5.0.0,>=3.4.1, but you have datasets 5.1.0 which is incompatible.
unsloth-zoo 202

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

C:\Users\sharm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sharm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\triton\windows_utils.py:465: UserWarning: Failed to find CUDA. Install CUDA Toolkit (https://developer.nvidia.com/cuda-downloads) or set CUDA_HOME / CUDA_PATH environment variable.
  warnings.warn("Failed to find CUDA."
C:\Users\sharm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.5.4)
  from 

In [3]:
MODEL_NAME = "google/gemma-2-9b-it"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [5]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

C:\Users\sharm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sharm\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\shar

In [6]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj","up_proj","down_proj"
],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
lora_adapter_model = get_peft_model(base_model, lora_config)

In [ ]:
lora_adapter_model.print_trainable_parameters()

In [ ]:
total_params = lora_adapter_model.num_parameters()
trainable_params = lora_adapter_model.num_parameters(only_trainable=True)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable %:", 100 * trainable_params / total_params)

In [ ]:
print(lora_adapter_model.peft_config)